# SENSEI — Session Intelligence
## Notebook 02: Sessionisation & Feature Engineering

The raw RetailRocket dataset is an **event log** — one row per user action.
For machine learning we need **one row per observation**.

This notebook transforms the event log into a session-level feature table:
one row per session, with engineered features as input columns and `purchased` as the target.

## Contents
1. Load the event log
2. Assign session IDs (sessionisation)
3. Session-level statistics
4. Feature engineering
5. Feature distributions
6. Class balance
7. Correlation with target
8. Export

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join('..', 'src'))
from session_utils import load_events, assign_session_ids, build_session_features

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = os.path.join('..', 'data')

## 1. Load the Event Log

We use `load_events()` from `src/session_utils.py`, which:
- reads the CSV
- parses the Unix millisecond timestamp to a proper datetime
- sorts by `visitorid` then `timestamp` (required for correct session assignment)

In [ ]:
df = load_events(os.path.join(DATA_DIR, 'events.csv'))
print(f'Rows   : {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head()

df = assign_session_ids(df)

n_sessions = df['session_id'].nunique()
n_visitors = df['visitorid'].nunique()
print(f'Events         : {len(df):,}')
print(f'Sessions       : {n_sessions:,}')
print(f'Visitors       : {n_visitors:,}')
print(f'Sessions/visitor (mean): {n_sessions / n_visitors:.2f}')

# Show an example: all events of one visitor, with session IDs assigned
example_visitor = df[df['visitorid'] == df['visitorid'].unique()[5]]
print(f'\nExample — visitor {example_visitor["visitorid"].iloc[0]}:')
example_visitor[['timestamp', 'event', 'itemid', 'session_id']].head(10)

## 3. Session-Level Statistics

Before building features, it helps to understand the distribution of session lengths.
Most sessions are very short (1–2 events). A long tail exists but is less common.

events_per_session = df.groupby('session_id').size()
print('Events per session:')
print(events_per_session.describe().round(1))
print(f'\nSessions with exactly 1 event: {(events_per_session == 1).sum():,} ({(events_per_session == 1).mean()*100:.1f} %)')

In [ ]:
events_per_session = df.groupby('session_id').size()

print('Events per session:')
print(events_per_session.describe().round(1))

## 4. Feature Engineering

We aggregate each session into a single row with numeric features.
`build_session_features()` in `src/session_utils.py` handles this.

**Features and their rationale:**

| Feature | Type | Why it matters |
|---|---|---|
| `n_views` | int | Raw engagement — how many products the visitor looked at |
| `n_addtocart` | int | Direct purchase signal — adding to cart indicates intent |
| `n_items` | int | Breadth of interest — browsing many vs. few items |
| `n_revisited_items` | int | Returning to the same item suggests stronger intent |
| `duration_sec` | float | Overall session length — highly engaged sessions last longer |
| `hour_of_day` | int | Purchase intent varies by time of day |
| `day_of_week` | int | Weekday vs. weekend browsing behaviour differs |
| `view_to_cart_ratio` | float | n_addtocart / n_views — how decisive the visitor is |
| `is_first_session` | int | First-time visitors behave differently from returning ones |

**What we deliberately excluded:**
- `n_events` — it equals `n_views + n_addtocart + n_transactions` exactly. Including it adds no new information and creates multicollinearity.
- `has_addtocart` — it is 100 % derivable from `n_addtocart > 0`. One of them is redundant.

sessions = build_session_features(df)
print(f'Shape: {sessions.shape}  →  {sessions.shape[0]:,} sessions × {sessions.shape[1]} columns')
sessions.head()

In [ ]:
# Descriptive statistics for all numeric features
# Note the high std and max values — most features are heavily right-skewed
sessions.describe().round(2)

FEATURES = [
    'n_views', 'n_addtocart', 'n_items', 'n_revisited_items',
    'duration_sec', 'hour_of_day', 'day_of_week',
    'view_to_cart_ratio', 'is_first_session',
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, FEATURES):
    data = sessions[col].clip(upper=sessions[col].quantile(0.99))
    sns.histplot(data, bins=40, ax=ax, color='#4C72B0')
    ax.set_title(col)
    ax.set_xlabel('')
    ax.set_ylabel('Sessions')

plt.suptitle('Feature distributions (clipped at 99th percentile for display)', y=1.01)
plt.tight_layout()
plt.show()

## 6. Class Balance

The target column `purchased` is binary:
- **0** — the session did not end in a purchase
- **1** — the session contained at least one transaction event

With ~0.81 % positive class, this is a heavily imbalanced classification problem.
A naive classifier that always predicts "no purchase" would achieve 99.19 % accuracy —
which is why accuracy is not a useful metric here. We will use **PR-AUC** and **F1** instead.

The bars below are on a **log scale** because on a linear scale the minority class bar
is barely a pixel tall at this ratio (1:122).

In [ ]:
features = ['n_events', 'n_views', 'n_addtocart', 'n_items', 'duration_sec']

fig, axes = plt.subplots(1, len(features), figsize=(18, 4))
for ax, col in zip(axes, features):
    data = sessions[col].clip(upper=sessions[col].quantile(0.99))
    sns.histplot(data, bins=40, ax=ax, color='#4C72B0')
    ax.set_title(col)
    ax.set_xlabel('')

plt.suptitle('Feature distributions (99th-percentile clipped)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Class Balance & Export

In [ ]:
# Point-biserial correlation between each feature and the binary target.
# This is mathematically equivalent to Pearson when one variable is binary,
# but it's worth noting the limitation: it only captures linear relationships.
# A feature with low Pearson correlation can still be useful in a tree-based model.

corr = (
    sessions[FEATURES + ['purchased']]
    .corr()['purchased']
    .drop('purchased')
    .sort_values(ascending=False)
)
print('Correlation with target (purchased):')
print(corr.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#55A868' if v > 0 else '#C44E52' for v in corr.values]
sns.barplot(x=corr.values, y=corr.index, ax=ax, palette=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Pearson correlation with target (purchased)')
ax.set_xlabel('Correlation coefficient')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

out_path = os.path.join(DATA_DIR, 'sessions_features.parquet')
sessions.to_parquet(out_path, index=False)

n_features = len(FEATURES)
print(f'Saved   : {out_path}')
print(f'Sessions: {len(sessions):,}')
print(f'Features: {n_features}  →  {FEATURES}')
print(f'Target  : purchased  (1 = {sessions["purchased"].sum():,} sessions, {sessions["purchased"].mean()*100:.2f} %)')

## Summary

| | |
|---|---|
| Sessions | ~1,761,675 |
| Features | 9 |
| Target | `purchased` — binary (0/1) |
| Purchase rate | 0.81 % — imbalance ratio ~1:122 |
| Output file | `data/sessions_features.parquet` |

**Next:** Notebook 03 loads this file and trains classification models.

In [ ]:
# Correlation with target
corr = sessions[features + ['has_addtocart', 'purchased']].corr()['purchased'].drop('purchased').sort_values(ascending=False)
print('Correlation with target (purchased):')
print(corr.round(3))

In [ ]:
# Export
out_path = os.path.join(DATA_DIR, 'sessions_features.parquet')
sessions.to_parquet(out_path, index=False)
print(f'Saved to {out_path}')
print(f'Final shape: {sessions.shape}  →  {sessions.shape[1]-1} features + 1 target')

## Summary

| Metric | Value |
|--------|-------|
| Total sessions | ~1,761,675 |
| Features | 9: `n_views`, `n_addtocart`, `n_items`, `n_revisited_items`, `duration_sec`, `hour_of_day`, `day_of_week`, `view_to_cart_ratio`, `is_first_session` |
| Target | `purchased` (binary) |
| Purchase rate | **0.81 %** — strongly imbalanced (ratio ~1:122) |

**Removed vs. original design:**
- `n_events` removed — strict linear combination of other features, adds multicollinearity
- `has_addtocart` removed — 100 % redundant with `n_addtocart`

**Next step → notebook 03: Classification with imbalanced data handling**